# Buoy raw data processing — spectra

Read the raw MHL directional spectral data (Sydney, Byron Bay, Batemans Bay
— see `00_Buoys_Processing.ipynb` for the bulk-parameter equivalent covering
all 35 buoys) and write **one NetCDF per station** to
`inputs/Observations/processed/spectra/`.

Format and parsing logic live in
[`utils/observation/raw_spectra.py`](utils/observation/raw_spectra.py) — see
its docstring for the two raw file types (yearly peak-normalized spectra +
the paired `*SD.TXT` absolute-density scale file) and the local-time-to-UTC
conversion applied here.

**Sydney's `SYDNEYSD.TXT` is corrupted** in the raw archive (entirely null
bytes) — Sydney's output has the normalized spectral shape only, not an
absolute energy density. Byron Bay and Batemans Bay both have valid SD
files and get both.

**The raw data has no full 2D E(f, dir) spectrum** — only one mean
direction and one directional spread *per frequency bin*, not a
per-direction energy breakdown. This notebook builds a proper
`(time, freq, dir)` spectrum from those two moments using the same
Cartwright (1963) cos-2s spreading function — and the same direction grid
(`utils.model.metamodel.SPECTRAL_DIR`, 5° bins) — that this codebase's own
metamodel spectral reconstruction already uses, so buoy and model spectra
are directly comparable. It's a **modelled** directional shape, not a
measurement; see the module docstring for why, and §2 below for a
conservation check.

In [1]:
from utils.config import resolve
from utils.observation import onshore_sites as obs
from utils.observation import raw_spectra as rs

## 1 · Read, join, pivot, and reconstruct the directional spectrum

For each station: concatenate all yearly files, drop a benign trailing
`freq=0` artifact row present in a few files (the real 64-bin hour next to
it is complete — see the module docstring), pivot to `(time, freq)`, join
the paired SD file's `spect_den` by timestamp, convert from local Eastern
Standard time (fixed UTC+10, no DST) to UTC, and spread each frequency
bin's energy over `SPECTRAL_DIR` using its own `dirn`/`spread` (Cartwright
cos-2s) to build the full `(time, freq, dir)` spectrum.

This reads ~1.3 GB of raw text and builds three ~2-3 GB in-memory spectral
cubes — a few minutes.

In [2]:
meta = obs.get_coordinates()
spectra_dir = resolve("inputs/Observations/processed/spectra")

datasets = {}
for iid in rs.STATIONS:
    ds = rs.read_station_spectra(iid)
    datasets[iid] = ds
    has_density = "efth" in ds
    print(
        f"{iid}: {ds.sizes['time']} hours, {ds.sizes['freq']} freq bins, "
        f"{ds.sizes['dir']} dir bins, "
        f"efth (absolute)={'yes' if has_density else 'NO (SD file corrupted)'}"
    )

Sydney_Buoy: 101449 hours, 64 freq bins, 72 dir bins, efth (absolute)=NO (SD file corrupted)


ByronBay_Buoy: 77163 hours, 64 freq bins, 72 dir bins, efth (absolute)=yes


BatemansBay_Buoy: 83733 hours, 64 freq bins, 72 dir bins, efth (absolute)=yes


## 2 · Sanity check against the bulk data

Cross-validate: reconstruct `Hm0 = 4*sqrt(m0)` from each hour's spectrum
(`m0` = spectrum integrated over frequency) and compare against the bulk
`hs` already processed in `00_Buoys_Processing.ipynb`, for the two stations
with an absolute `energy_density`. They're different estimators (spectral
Hm0 vs. statistical Hsig) so they won't match exactly, but should track
closely — this catches join/unit/timezone mistakes, not just typos.

In [3]:
import numpy as np
import pandas as pd
import xarray as xr

for iid in ["ByronBay_Buoy", "BatemansBay_Buoy"]:
    ds = datasets[iid]
    bulk = xr.open_dataset(resolve(f"inputs/Observations/processed/{iid}.nc"))

    freq = ds.freq.values
    df_bins = np.where(freq <= 0.1 + 1e-9, 0.005, 0.01)  # raw bin width: 0.005 up to 0.1 Hz, 0.01 above
    m0 = np.nansum(ds["energy_density"].values * df_bins, axis=1)
    hm0 = 4 * np.sqrt(m0)

    bulk_hs = bulk["hs"].reindex(time=ds.time, method="nearest", tolerance=pd.Timedelta("2min"))
    ratio = hm0 / bulk_hs.values
    valid = np.isfinite(ratio) & (bulk_hs.values > 0.2)
    outliers = valid & ((ratio > 2.5) | (ratio < 0.4))

    print(f"{iid}: {valid.sum()} hourly comparisons, "
          f"{outliers.sum()} ({100 * outliers.sum() / valid.sum():.2f}%) "
          f"disagree by >2.5x — isolated raw spect_den glitches, not filtered here")

ByronBay_Buoy: 77015 hourly comparisons, 8 (0.01%) disagree by >2.5x — isolated raw spect_den glitches, not filtered here


BatemansBay_Buoy: 83620 hourly comparisons, 7 (0.01%) disagree by >2.5x — isolated raw spect_den glitches, not filtered here


## 3 · Directional reconstruction check

The Cartwright spreading function is built to conserve energy: integrating
`efth` back over `dir` should exactly reproduce `energy_density` (same
check for `efth_norm` against `energy_norm`) — this isn't a physical
validation of the modelled shape, just confirmation the construction and
grid are wired up correctly.

In [4]:
dir_step = float(datasets["Sydney_Buoy"]["dir"].diff("dir").isel(dir=0))  # SPECTRAL_DIR spacing, degrees

for iid, ds in datasets.items():
    reintegrated_norm = ds["efth_norm"].sum(dim="dir") * dir_step
    err_norm = float((reintegrated_norm - ds["energy_norm"]).max(skipna=True))
    line = f"{iid}: max |reintegrated efth_norm - energy_norm| = {err_norm:.2e}"
    if "efth" in ds:
        reintegrated = ds["efth"].sum(dim="dir") * dir_step
        err = float((reintegrated - ds["energy_density"]).max(skipna=True))
        line += f", max |reintegrated efth - energy_density| = {err:.2e}"
    print(line)

Sydney_Buoy: max |reintegrated efth_norm - energy_norm| = 6.36e-07


ByronBay_Buoy: max |reintegrated efth_norm - energy_norm| = 6.36e-07, max |reintegrated efth - energy_density| = 4.13e-05


BatemansBay_Buoy: max |reintegrated efth_norm - energy_norm| = 6.36e-07, max |reintegrated efth - energy_density| = 1.76e-05


## 4 · Write one NetCDF per station

In [5]:
rows = []
for iid in rs.STATIONS:
    out_path = rs.standardize_and_write(iid, meta[iid], spectra_dir)
    ds = datasets[iid]
    rows.append(
        {
            "instrument_id": iid,
            "n_hours": ds.sizes["time"],
            "n_freq": ds.sizes["freq"],
            "n_dir": ds.sizes["dir"],
            "period_start": str(ds.time.values.min())[:10],
            "period_end": str(ds.time.values.max())[:10],
            "has_efth": "efth" in ds,
            "file_size_mb": round(out_path.stat().st_size / 1e6),
            "file": out_path.name,
        }
    )

summary = pd.DataFrame(rows).set_index("instrument_id")
print(f"wrote {len(summary)} station spectra file(s) to {spectra_dir}")
summary

wrote 3 station spectra file(s) to /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/inputs/Observations/processed/spectra


,n_hours,n_freq,n_dir,period_start,period_end,has_efth,file_size_mb,file
instrument_id,,,,,,,,
Sydney_Buoy,101449,64,72,1998-12-31,2011-12-31,False,1526,Sydney_Buoy.nc
ByronBay_Buoy,77163,64,72,1999-10-26,2011-12-31,True,2267,ByronBay_Buoy.nc
BatemansBay_Buoy,83733,64,72,2001-02-23,2011-12-31,True,2530,BatemansBay_Buoy.nc


## 5 · Verify

In [6]:
for iid in rs.STATIONS:
    path = spectra_dir / f"{iid}.nc"
    ds = xr.open_dataset(path)
    print(f"=== {iid} ===")
    print(f"  vars  : {sorted(ds.data_vars)}")
    print(f"  dims  : {dict(ds.sizes)}")
    print(f"  period: {ds.time.values[0]} to {ds.time.values[-1]} (UTC)")
    ds.close()

=== Sydney_Buoy ===
  vars  : ['dirn', 'efth_norm', 'energy_norm', 'kurtosis', 'skewness', 'spread']
  dims  : {'time': 101449, 'freq': 64, 'dir': 72}
  period: 1998-12-31T14:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
=== ByronBay_Buoy ===
  vars  : ['dirn', 'efth', 'efth_norm', 'energy_density', 'energy_norm', 'kurtosis', 'skewness', 'spect_den', 'spread']
  dims  : {'time': 77163, 'freq': 64, 'dir': 72}
  period: 1999-10-26T07:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
=== BatemansBay_Buoy ===
  vars  : ['dirn', 'efth', 'efth_norm', 'energy_density', 'energy_norm', 'kurtosis', 'skewness', 'spect_den', 'spread']
  dims  : {'time': 83733, 'freq': 64, 'dir': 72}
  period: 2001-02-23T00:00:00.000000000 to 2011-12-31T13:00:00.000000000 (UTC)
